In [1]:
import numpy as np
import gzip
from pathlib import Path
import matplotlib.pyplot as plt
from tqdm import tqdm
import torch
import torch.nn as nn

import seaborn as sns


# Cargamos el dataset MNIST y lo particionamos

In [2]:
def load_mnist_dataset(mnist_path):
    x_trainval = get_images(Path(mnist_path)/Path('train-images-idx3-ubyte.gz'))
    y_trainval = get_labels(Path(mnist_path)/Path('train-labels-idx1-ubyte.gz'))

    x_train = x_trainval[:50000]
    y_train = y_trainval[:50000]

    x_val = x_trainval[50000:]
    y_val = y_trainval[50000:]

    x_test = get_images(Path(mnist_path)/Path('t10k-images-idx3-ubyte.gz'))
    y_test = get_labels(Path(mnist_path)/Path('t10k-labels-idx1-ubyte.gz'))

    return x_train, y_train, x_val, y_val, x_test, y_test

def get_labels(path):
    with gzip.open(path, 'rb') as data:
        labels = data.read()[8:]
        return np.frombuffer(labels, dtype=np.uint8)

def get_images(path):
    with gzip.open(path, 'rb') as data:
        _ = int.from_bytes(data.read(4), 'big')
        num_images = int.from_bytes(data.read(4), 'big')
        rows = int.from_bytes(data.read(4), 'big')
        cols = int.from_bytes(data.read(4), 'big')
        images = data.read()
        return np.frombuffer(images, dtype=np.uint8).reshape((num_images, rows, cols))


In [3]:
x_train, y_train, x_val, y_val, x_test, y_test = load_mnist_dataset('datasets/mnist')

In [4]:
x_train = x_train.copy().reshape(50000, -1).astype(np.float32)
y_train = y_train.copy().reshape(50000, 1)

x_val = x_val.copy().reshape(10000, -1).astype(np.float32)
y_val = y_val.copy().reshape(10000, 1)

x_test = x_test.copy().reshape(10000, -1).astype(np.float32)
y_test = y_test.copy().reshape(10000, 1)

def scale(x_mean, x_std, x_data):
    return (x_data - x_mean) / x_std

x_mean = x_train.mean()
x_std = x_train.std()

x_train = scale(x_mean, x_std, x_train)
x_val = scale(x_mean, x_std, x_val)
x_test = scale(x_mean, x_std, x_test)

In [5]:
from torch.utils.data import DataLoader, TensorDataset

x_train_tensor = torch.tensor(x_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.squeeze(), dtype=torch.long)
x_val_tensor = torch.tensor(x_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val.squeeze(), dtype=torch.long)
x_test_tensor = torch.tensor(x_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.squeeze(), dtype=torch.long)

dataset = TensorDataset(x_train_tensor, y_train_tensor)

dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

# Red FF

In [6]:
class FeedForwardNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.hidden = nn.Linear(input_size, hidden_size)
        self.output = nn.Linear(hidden_size, output_size)
        self.relu = nn.ReLU()
    def forward(self, x):
        x = self.hidden(x)
        x = self.relu(x)
        x = self.output(x)
        return x

In [7]:
first_model = FeedForwardNN(input_size=784, hidden_size=32, output_size=10)
second_model = FeedForwardNN(input_size=784, hidden_size=32, output_size=10)
third_model = FeedForwardNN(input_size=784, hidden_size=500, output_size=10)

# Función de pérdida

In [8]:
loss_fn = nn.CrossEntropyLoss()

# Optimizador

In [9]:
import torch.optim as optim

first_optimizer = optim.Adam(first_model.parameters(), lr=0.001)
second_optimizer = optim.SGD(second_model.parameters(), lr=0.001)
third_optimizer = optim.Adam(third_model.parameters(), lr=0.0015)

# Entrenamiento

In [10]:
def accuracy(model, x, y):
    model.eval()
    with torch.no_grad():
        logists = model(x)
        predicciones = torch.argmax(logists, dim=1)
        correctas = (predicciones == y).sum().item()
        total = y.shape[0]
    return correctas/total

In [11]:
from collections import deque # disclaimer: googleé este paquete porque hace más idiomático el uso del historial
def train(model, loss_fn, optimizer, num_epochs=64, paciencia=10, delta=0.005):
    historial_eval = deque(maxlen=paciencia+1)
    for epoch in range(num_epochs):
        model.train()
        for X_batch, y_batch in dataloader:
            predictions = model(X_batch)
            loss = loss_fn(predictions, y_batch) # ¿CÓMO QUE PYTHON NO USA BLOCK-LEVEL SCOPING PARA LOS FOR LOOPS?
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            eval_logists = model(x_val_tensor)
            eval_loss = loss_fn(eval_logists, y_val_tensor)
            eval_acc = accuracy(model, x_val_tensor, y_val_tensor)
            historial_eval.append(eval_loss.item())
        
        print(f"Época {epoch+1}")
        print(f"Pérdida al entrenar: {loss.item():.4f}")
        print(f"Exactitud al evaluar: {eval_acc:.4f}")
        print(f"Pérdida al evaluar: {eval_loss:.4f}")
        
        if len(historial_eval) > paciencia and 0 <= (historial_eval[0] - historial_eval[-1]) < delta:
            print("Se nos acaba la paciencia.") # yo cuando python
            break

In [12]:
train(model=first_model, loss_fn=loss_fn, optimizer=first_optimizer)
train(model=second_model, loss_fn=loss_fn, optimizer=second_optimizer)
train(model=third_model, loss_fn=loss_fn, optimizer=third_optimizer, num_epochs=12)

Época 1
Pérdida al entrenar: 0.2775
Exactitud al evaluar: 0.9382
Pérdida al evaluar: 0.2048
Época 2
Pérdida al entrenar: 0.0760
Exactitud al evaluar: 0.9576
Pérdida al evaluar: 0.1552
Época 3
Pérdida al entrenar: 0.1730
Exactitud al evaluar: 0.9598
Pérdida al evaluar: 0.1399
Época 4
Pérdida al entrenar: 0.2976
Exactitud al evaluar: 0.9585
Pérdida al evaluar: 0.1405
Época 5
Pérdida al entrenar: 0.0142
Exactitud al evaluar: 0.9607
Pérdida al evaluar: 0.1304
Época 6
Pérdida al entrenar: 0.0054
Exactitud al evaluar: 0.9600
Pérdida al evaluar: 0.1314
Época 7
Pérdida al entrenar: 0.0282
Exactitud al evaluar: 0.9624
Pérdida al evaluar: 0.1251
Época 8
Pérdida al entrenar: 0.0231
Exactitud al evaluar: 0.9678
Pérdida al evaluar: 0.1121
Época 9
Pérdida al entrenar: 0.0622
Exactitud al evaluar: 0.9644
Pérdida al evaluar: 0.1249
Época 10
Pérdida al entrenar: 0.0482
Exactitud al evaluar: 0.9613
Pérdida al evaluar: 0.1260
Época 11
Pérdida al entrenar: 0.0199
Exactitud al evaluar: 0.9601
Pérdida al ev

# Evaluación

In [13]:
def eval(model):
    model.eval()
    with torch.no_grad():
        test_logists = model(x_test_tensor)
        test_loss = loss_fn(test_logists, y_test_tensor)
        test_predicciones = torch.argmax(test_logists, dim=1)
        confucio = torch.zeros(10, 10, dtype=torch.int64)
        for tensori, predictori in zip(y_test_tensor, test_predicciones):
            confucio[tensori,predictori] = 1 + confucio[tensori,predictori]
        
        exactitud = torch.diag(confucio).sum().item() / confucio.sum().item()
        
        precisión, recall, f1 = (torch.zeros(10), torch.zeros(10), torch.zeros(10))
        
        for i in range(10):
            atinado = confucio[i,i].item()
            falsos_neg = confucio[i,:].sum().item() - atinado
            falsos_posi = confucio[:,i].sum().item() - atinado
            
            recall[i] = atinado/(atinado+falsos_neg)
            precisión[i] = atinado / (atinado+falsos_posi)
            f1[i] = 2 * (precisión[i]*recall[i])/(precisión[i]+recall[i])
            
        print(f"Pérdida en prueba: {test_loss.item():.4f}")
        print(f"Exactitud en prueba: {exactitud:.4f}")
        print(f"Matriz de confusión")
        print(confucio)
        print("Métricas para cada clase")
        for i in range(10):
            print(f"\tClase {i}")
            print(f"\t\tRecall: {recall[i]:.4f}")
            print(f"\t\tF1: {f1[i]:.4f}")
            print(f"\t\tPrecisión: {precisión[i]:.4f}")
        print("\n")

In [14]:
print("Primer modelo")
eval(model=first_model)
print("Segundo modelo")
eval(model=second_model)
print("Tercer modelo")
eval(model=third_model)

Primer modelo
Pérdida en prueba: 0.1316
Exactitud en prueba: 0.9616
Matriz de confusión
tensor([[ 950,    1,    9,    1,    0,    5,    6,    3,    3,    2],
        [   0, 1122,    3,    4,    1,    1,    3,    0,    1,    0],
        [   3,    3, 1008,   10,    0,    0,    1,    2,    4,    1],
        [   0,    0,   15,  978,    0,    7,    0,    4,    4,    2],
        [   0,    2,   11,    1,  941,    0,    4,    6,    0,   17],
        [   2,    1,    2,   25,    3,  833,    7,    2,   10,    7],
        [   2,    4,    2,    2,    8,    4,  930,    0,    5,    1],
        [   0,    5,   19,   12,    2,    1,    0,  978,    2,    9],
        [   1,    2,   10,   25,    7,    2,    0,    6,  912,    9],
        [   1,    4,    1,   14,   12,    2,    0,   10,    1,  964]])
Métricas para cada clase
	Clase 0
		Recall: 0.9694
		F1: 0.9799
		Precisión: 0.9906
	Clase 1
		Recall: 0.9885
		F1: 0.9846
		Precisión: 0.9808
	Clase 2
		Recall: 0.9767
		F1: 0.9545
		Precisión: 0.9333
	Clase 3
